In [ ]:
import sys
import os
from dotenv import load_dotenv
import pandas as pd
from datasets import Dataset

_PROJECT_ROOT = os.path.abspath("..")
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
load_dotenv(os.path.join(_PROJECT_ROOT, ".env"))

from src.cache import clear_cache
from src.main import process_query, setup_observability
from src import config

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.run_config import RunConfig

# Start Phoenix first (do not use nest_asyncio here — it breaks Phoenix on Python 3.14)
setup_observability()

eval_llm = ChatGroq(model=config.LLM_MODEL, temperature=0)
eval_embeddings = HuggingFaceEmbeddings(
    model_name=config.EMBEDDING_MODEL,
    model_kwargs={"device": config.EMBEDDING_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

ragas_llm = LangchainLLMWrapper(eval_llm)
ragas_emb = LangchainEmbeddingsWrapper(eval_embeddings)

MAX_CONTEXT_CHARS = 3000  # long web-search context slows faithfulness a lot


def truncate_contexts(contexts: list[str]) -> list[str]:
    out = []
    for text in contexts:
        if len(text) > MAX_CONTEXT_CHARS:
            out.append(text[:MAX_CONTEXT_CHARS] + "...")
        else:
            out.append(text)
    return out


eval_questions = [
    "What is the concept of Self-RAG?",
    "How does Self-RAG handle hallucinations?",
    "What is the weather in Tokyo?",
]

ground_truths = [
    "Self-RAG is a framework that retrieves relevant passages on-demand, and uses self-reflection to critique and select the best outputs.",
    "It uses a self-reflection mechanism with critic models to evaluate if the generated text is grounded in the retrieved facts.",
    "I do not have real-time weather information in my local database, but a web search shows the current weather in Tokyo.",
]

clear_cache()

print("Generating answers via SentinelRAG (cache bypassed for eval)...")
agentic_answers = []
retrieved_contexts = []
for q in eval_questions:
    result = process_query(q, use_cache=False)
    agentic_answers.append(result["answer"])
    retrieved_contexts.append(
        truncate_contexts(result["contexts"] or ["No context retrieved."])
    )

data = {
    "question": eval_questions,
    "answer": agentic_answers,
    "ground_truth": ground_truths,
    "contexts": retrieved_contexts,
}
dataset = Dataset.from_dict(data)

print("\nRunning Ragas Evaluation...")
print("6 runs total (3 questions × 2 metrics). Each metric calls Groq several times.")
print("Expect several minutes — progress bar may sit at 0% while the first call runs.")

result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=RunConfig(
        max_workers=1,   # avoid Groq rate-limit retries that look like a hang
        max_retries=5,
        timeout=120,
    ),
    allow_nest_asyncio=True,
)

df = result.to_pandas()
print("\n=== EVALUATION RESULTS ===")
print(df[["question", "faithfulness", "answer_relevancy"]])

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/ramunalla/Personal/sentinel-RAG/src/cache.py:26: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to s

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/ramunalla/Personal/sentinel-RAG/src/nodes.py:48: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(k=3)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please swit

...

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/gc/5sc3n9hs70bgbch5c1skcqbr0000gp/T/ipykernel_59332/381636602.py:33: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(eval_llm)
/var/folders/gc/5sc3n9hs70bgbch5c1skcqbr0000gp/T/ipykernel_59332/381636602.py:34: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.emb

--- CACHE CLEARED (1 entries removed) ---
Generating answers via SentinelRAG (cache bypassed for eval)...

[USER QUERY]: What is the concept of Self-RAG?

--- INITIATING AGENTIC WORKFLOW ---
---NODE: RETRIEVE FROM VECTOR DB---
---NODE: GRADE DOCUMENT RELEVANCE---
  - GRADE: Document Relevant
  - GRADE: Document Relevant
  - GRADE: Document Relevant
---EDGE: EVALUATE RETRIEVAL RESULTS---
  - DECISION: Documents relevant. Routing to Generate.
---NODE: GENERATE ANSWER---


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openinference/instrumentation/_spans.py:43: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  masked_value = self._self_config.mask(key, value)


---EDGE: CHECK FOR HALLUCINATIONS---
  - DECISION: Answer is Grounded (No Hallucination).
---EDGE: CHECK IF ANSWER RESOLVES QUERY---
  - DECISION: Answer is Useful and Resolves Query.

[FINAL ANSWER FROM AGENT]:
Self-RAG is a concept that involves a retriever model retrieving relevant passages, then concurrently processing and evaluating them to generate task outputs. It also critiques its own output and chooses the best one based on factuality and quality, allowing for easier fact verification.

🐢 [LATENCY]: 2.2599 seconds 🐢

[USER QUERY]: How does Self-RAG handle hallucinations?

--- INITIATING AGENTIC WORKFLOW ---
---NODE: RETRIEVE FROM VECTOR DB---
---NODE: GRADE DOCUMENT RELEVANCE---
  - GRADE: Document Relevant
  - GRADE: Document Irrelevant (REJECTED)
  - GRADE: Document Relevant
---EDGE: EVALUATE RETRIEVAL RESULTS---
  - DECISION: Documents relevant. Routing to Generate.
---NODE: GENERATE ANSWER---
---EDGE: CHECK FOR HALLUCINATIONS---
  - DECISION: Answer is Grounded (No Halluc

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]